# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khizertouseef76-hue/Flyrank_Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row = One unique content page (content_id).  Time Window: A 90-day historical performance window prior to the decision/review point.  Verification Plan: Verify that content_id has zero duplicate rows and inspect the date/age span of content across the dataset


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np

# 1. Colab & Directory Setup
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_DIR = "Flyrank_ML_Internship"
    REPO_URL = f"https://github.com/khizertouseef76-hue/{REPO_DIR}.git"
    if not os.path.isdir(f"/content/{REPO_DIR}"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, f"/content/{REPO_DIR}"], check=True)
    os.chdir(f"/content/{REPO_DIR}")

while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 2. Verify Grain & Time Window
total_rows = len(df)
unique_ids = df["content_id"].nunique()

print(f"Total rows in dataset: {total_rows:,}")
print(f"Unique content_ids:    {unique_ids:,}")
assert total_rows == unique_ids, "Grain Check Failed: Duplicates exist!"
print("Result: Grain verified — exactly 1 row = 1 unique content page.\n")

print("Time Window Check (Content Age Span):")
print(f"Minimum content age: {df['content_age_days'].min()} days")
print(f"Maximum content age: {df['content_age_days'].max()} days")

Total rows in dataset: 30,000
Unique content_ids:    30,000
Result: Grain verified — exactly 1 row = 1 unique content page.

Time Window Check (Content Age Span):
Minimum content age: 90 days
Maximum content age: 564 days


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Data Contract Field CategorizationFeatures (Pre-decision observable signals):impressions_90d: Total Google Search impressions in the prior 90 days.  clicks_90d: Total Google Search clicks in the prior 90 days.  ctr: Click-Through Rate (clicks_90d / impressions_90d).  avg_position: Average Google search position.  days_since_last_update: Days elapsed since content was last modified.  word_count: Total word length of the article.  content_age_days: Total days since initial publication.  Label (Target Proxy):is_declining_label: Binary flag derived as (trend_direction == "down").  Context (Grouping & Metadata):content_id, client_id: Pseudonymized IDs for grouping and client-holdout splitting.  content_type, main_intent: Categorical content metadata.  Excluded (With Reasons):trend_pct: Leaky Feature — exact percentage change from which trend_direction is directly calculated. Including it causes data leakage.  Product Decision Flags / App Scores (health_score, priority_score): Excluded to ensure the model discovers signals independently rather than copying app rules.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define explicit feature set and label
feature_cols = [
    "impressions_90d", "clicks_90d", "ctr",
    "avg_position", "days_since_last_update",
    "word_count", "content_age_days"
]
target_col = "is_declining_label"

# Create binary label
df[target_col] = (df["trend_direction"].str.lower() == "down").astype(int)

print("Selected Features Frame Shape:", df[feature_cols].shape)
print("\nTarget Label Balance (1 = Declining, 0 = Healthy/Up/Flat):")
print(df[target_col].value_counts(normalize=True).round(3))

Selected Features Frame Shape: (30000, 7)

Target Label Balance (1 = Declining, 0 = Healthy/Up/Flat):
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification QueriesWe run four explicit queries to prove our contract claims:  Grain Uniqueness: Verify 0 duplicate content_ids.  Missing Values Audit: Check missingness across all features.  Availability Filter (IS TRUE): Count rows surviving a minimum visibility threshold (impressions_90d >= 100 IS TRUE).  Label Rate: Check declining class proportion across visibility tiers.  

In [3]:
# Query 1: Grain Uniqueness
duplicates = df["content_id"].duplicated().sum()
print(f"Query 1 (Grain Uniqueness): {duplicates} duplicate IDs found.")

# Query 2: Missing Values Audit
missing_counts = df[feature_cols].isnull().sum()
print("\nQuery 2 (Missing Values Check):")
print(missing_counts)

# Query 3: Availability Filter Check (impressions >= 100 IS TRUE)
df["visible_100"] = (df["impressions_90d"] >= 100)
surviving = (df["visible_100"] == True).sum()
print(f"\nQuery 3 (Availability Filter): {surviving:,} of {len(df):,} rows ({surviving/len(df):.1%}) survive 'impressions_90d >= 100 IS TRUE'.")

# Query 4: Target Rate Comparison
all_decline_rate = df[target_col].mean()
vis_decline_rate = df[df["visible_100"] == True][target_col].mean()
print(f"\nQuery 4 (Label Rate): Overall = {all_decline_rate:.3f} | Visible Pages (>=100 imp) = {vis_decline_rate:.3f}")

Query 1 (Grain Uniqueness): 0 duplicate IDs found.

Query 2 (Missing Values Check):
impressions_90d              0
clicks_90d                   0
ctr                          0
avg_position                 0
days_since_last_update       0
word_count                7699
content_age_days             0
dtype: int64

Query 3 (Availability Filter): 22,006 of 30,000 rows (73.4%) survive 'impressions_90d >= 100 IS TRUE'.

Query 4 (Label Rate): Overall = 0.542 | Visible Pages (>=100 imp) = 0.598


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data LimitationsProxy Target vs. Future Window: The target label (trend_direction == "down") is computed over the historical observation window rather than a separate future target window. In production, features must come from a prior window and labels from a future window.  Low-Volume Volatility: Pages with very low impressions (< 100) have high CTR variance. Filtering for minimum volume is required to avoid fitting model rules on random noise.  Observational Boundary: Search performance signals describe what occurred historically; they do not prove causation or internal Google ranking mechanics.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Code proof: High variance in low-volume pages vs stability in high-volume pages
low_vol_std = df[df["impressions_90d"] < 50]["ctr"].std()
high_vol_std = df[df["impressions_90d"] >= 500]["ctr"].std()

print(f"CTR Std Dev (Low Volume < 50 imp):   {low_vol_std:.4f}")
print(f"CTR Std Dev (High Volume >= 500 imp): {high_vol_std:.4f}")
print("Proof that low-volume pages introduce noise, requiring minimum-volume filtering.")

CTR Std Dev (Low Volume < 50 imp):   6.9261
CTR Std Dev (High Volume >= 500 imp): 0.3169
Proof that low-volume pages introduce noise, requiring minimum-volume filtering.


## Self-check

Before you submit, confirm each line honestly:

- [Done ] Every section above is filled — markdown thinking AND the code that backs it
- [Done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done ] No client names, URLs, or private queries anywhere
- [Done ] My claims use careful words: observed, measured, directional, decision-support
- [Done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.